In [ ]:
from pathlib import Path
import json
import pandas as pd

In [ ]:
project_dir = Path.cwd()
data_dir = project_dir / "data"

print("Project folder:", project_dir)
print("Data folder exists:", data_dir.exists())

In [ ]:
for file in sorted(data_dir.rglob("*")):
    if file.is_file():
        print(file.relative_to(data_dir))

In [ ]:
with open(data_dir / "reporting_config.json", encoding="utf-8") as file:
    config = json.load(file)

print(json.dumps(config, indent=4))    # json string

In [ ]:
print("First week:", config["reporting_window"]["first_week_start"])
print("Last week:", config["reporting_window"]["last_week_start"])
print("Required division:", config["scope"]["division"])

**loading Data - Hr Roster**

**Data Inspection**

In [ ]:
hr = pd.read_csv(data_dir / "hr_roster.csv")    #loading Hr Roter data set

hr.head()

**Data Inspection:-**

In [ ]:
print("Rows and columns:", hr.shape)
print("Unique employees:", hr["Employee_ID"].nunique())

hr.info()

In [ ]:
print("Rows and columns:", hr.shape)                         # how many rows & columns
print("Unique employees:", hr["Employee_ID"].nunique())

hr.info()

In [ ]:
print("Missing values by column:")
print(hr.isna().sum())

print("\nExact duplicate rows:", hr.duplicated().sum())              # every column matches another row

In [ ]:
multiple_records = hr[hr.duplicated(subset="Employee_ID", keep=False)]   # keep=False = marks all occurrences of a repeated employee ID

multiple_records.sort_values(["Employee_ID", "Effective_Date"]).head(10)

## HR data preparation
Standardise employee IDs, select each employee's latest HR record,
and retain employees in the Advisory division.

In [ ]:
# Keep the original HR data unchanged for comparison.

hr_clean = hr.copy()

In [ ]:
# Remove surrounding spaces and convert IDs to uppercase.

hr_clean["Employee_ID"] = hr_clean["Employee_ID"].str.strip().str.upper()


hr_clean.head()

In [ ]:
# Convert text dates into actual dates so we can order records correctly.
# The source format is year-month-day, for example 2026-08-14.

hr_clean["Effective_Date"] = pd.to_datetime(hr_clean["Effective_Date"],format="%Y-%m-%d")

# Check that Effective_Date now has a datetime type.
hr_clean.dtypes

In [ ]:
# Group each employee's records together and order them by date.

hr_sorted = hr_clean.sort_values(by=["Employee_ID", "Effective_Date"])

hr_sorted.head(10)

In [ ]:
# Because dates are sorted oldest to newest, keep="last"
# retains the most recent record for each employee.

hr_latest = hr_sorted.drop_duplicates(subset="Employee_ID",keep="last")

print("Rows before selecting latest records:", len(hr_sorted))
print("Rows after selecting latest records:", len(hr_latest))

Here, drop_duplicates() checks only Employee_ID because we specified subset. We are deliberately selecting one record per employee—not claiming their historical records were mistakes.

In [ ]:
# Read the required division from the configuration.
required_division = config["scope"]["division"]

# Filter only AFTER selecting each employee's latest record.
hr_advisory = hr_latest[hr_latest["Division"] == required_division].copy()

print("Eligible Advisory employees:", len(hr_advisory))

hr_advisory.head()

question - Why this order? If someone moved from ADV to TAX, their latest TAX record must determine eligibility. Filtering to ADV first would wrongly retain their old record.

In [ ]:
# Each employee should appear only once.
print("Repeated employee IDs:", hr_advisory["Employee_ID"].duplicated().sum())

# Every remaining record should belong to ADV.
print("Remaining divisions:", hr_advisory["Division"].unique())

In [ ]:
hr_advisory["Division"].value_counts()

## Quill usage data
Explore the event log and prepare it for weekly user calculations.

In [ ]:
# Load the Quill event log.
quill = pd.read_csv(data_dir / "quill_events.csv")

quill.head()

In [ ]:
# Check the number of rows, columns and their data types.
print("Rows and columns:", quill.shape)

quill.info()

In [ ]:
# Count how many records belong to each type of action.

quill["event_type"].value_counts()

In [ ]:
# Check for missing values in each column.

quill.isna().sum()

generation = counted towards Quill’s active and power-user thresholds

open & no-generation = not an active user


In [ ]:
# Preserve the original data and work on a copy.

quill_clean = quill.copy()

In [ ]:
quill_clean.head()

In [ ]:
# Standardise IDs so they match the format used in HR.

quill_clean["user_id"] = quill_clean["user_id"].str.strip().str.upper()

In [ ]:
quill_clean["user_id"].head(2)

In [ ]:
# Count records where the ID formatting changed.

changed_ids = (quill["user_id"] != quill_clean["user_id"]).sum()

print("Rows with ID formatting changes:", changed_ids)

In [ ]:
# Count exact duplicate rows in the original file.
print("Raw duplicate rows:", quill.duplicated().sum())

In [ ]:
# Check again after standardising IDs.
print("Duplicate rows after ID cleaning:",quill_clean.duplicated().sum())

In [ ]:
# Display all occurrences of duplicate rows so we can inspect them.

duplicate_events = quill_clean[quill_clean.duplicated(keep=False)]

duplicate_events.sort_values(["user_id", "event_time_utc"]).head(10)

In [ ]:
# Convert event timestamps from text to UTC datetime values.

quill_clean["event_time_utc"] = pd.to_datetime(quill_clean["event_time_utc"],format="%Y-%m-%dT%H:%M:%SZ",utc=True,errors="raise")

# Check the converted column type and sample values.
print(quill_clean["event_time_utc"].dtype)
quill_clean["event_time_utc"].head()

In [ ]:
# Calculate the number of days since Sunday for each event.

days_since_sunday = (quill_clean["event_time_utc"].dt.dayofweek + 1) % 7

# Display the first five results.
days_since_sunday

In [ ]:
# Assign each event to its Sunday week start at midnight UTC.
quill_clean["week_start"] = (quill_clean["event_time_utc"].dt.normalize() - pd.to_timedelta(days_since_sunday, unit="D"))

# Compare event timestamps with their reporting week starts.
quill_clean[["event_time_utc", "week_start"]].head(10)

In [ ]:
# Count available event rows for each reporting week.
quill_weekly_coverage = quill_clean.groupby("week_start").size()

# Display weekly event counts in date order.
quill_weekly_coverage.sort_index()

**Reporting period**

In [ ]:
# Read the reporting boundaries from the configuration as UTC datetimes.
first_week = pd.to_datetime(config["reporting_window"]["first_week_start"], utc=True)

last_week = pd.to_datetime(config["reporting_window"]["last_week_start"], utc=True)

# Keep events belonging to the requested reporting weeks.
quill_window = quill_clean[(quill_clean["week_start"] >= first_week) & (quill_clean["week_start"] <= last_week)].copy()

# Check the remaining weeks and their event counts.
quill_window.groupby("week_start").size()

In [ ]:
# Keep events belonging to employees in the latest Advisory roster.
quill_advisory = quill_window[quill_window["user_id"].isin(hr_advisory["Employee_ID"])].copy()

# Check the number of events and distinct users retained.
print("Advisory event rows:", len(quill_advisory))
print("Distinct Advisory users:", quill_advisory["user_id"].nunique())

In [ ]:
# Keep generation events used to calculate Quill activity thresholds.
quill_generations = quill_advisory[quill_advisory["event_type"] == "generation"].copy()

# Check the retained event type and number of rows.
print(quill_generations["event_type"].value_counts())

In [ ]:
# Count generation event rows for each employee in each week.
# Duplicate records are still included at this stage.
quill_user_week = (quill_generations.groupby(["week_start", "user_id"]).size().reset_index(name="generation_count"))

# Display the first ten employee-week totals.
quill_user_week.head(10)

In [ ]:
quill_user_week.tail(10)

In [ ]:
# Create a comparison dataset with one copy of each identical event.

quill_generations_dedup = quill_generations.drop_duplicates(subset=["user_id", "event_time_utc", "event_type", "client"], keep="first")

quill_generations_dedup

In [ ]:
# Count weekly generations after removing identical event copies.

quill_user_week_dedup = (quill_generations_dedup.groupby(["week_start", "user_id"]).size().reset_index(name="generation_count_dedup"))
quill_user_week_dedup.head(10)

In [ ]:
# Compare counts with and without identical event copies.

quill_duplicate_comparison = quill_user_week.merge(quill_user_week_dedup,on=["week_start", "user_id"],how="left",validate="one_to_one")
quill_duplicate_comparison

In [ ]:
# Display employee-weeks where the generation count changed.

quill_duplicate_comparison[quill_duplicate_comparison["generation_count"]!= quill_duplicate_comparison["generation_count_dedup"]].head(10)

13 → 12: dono cases mein power user.
16 → 14: dono cases mein power user.
11 → 9: dono cases mein power user nahi.
12 → 11 hota: tab classification change hoti—pehle power user, baad mein nahi.

In [ ]:
# Read the Quill power-user threshold from the configuration.
for tool in config["tools"]:
    if tool["tool_id"] == "quill":
        quill_power_threshold = tool["wpu_threshold"]

# Identify employee-weeks that fall below the threshold after deduplication.
power_status_changed = quill_duplicate_comparison[(quill_duplicate_comparison["generation_count"] >= quill_power_threshold)
    & (quill_duplicate_comparison["generation_count_dedup"]< quill_power_threshold ) ].copy()

# Show the threshold and the number of affected employee-weeks.
print("Power-user threshold:", quill_power_threshold)
print("Employee-weeks with changed power status:", len(power_status_changed))

power_status_changed.head(10)

In [ ]:
# Count changed power-user classifications in each affected week.
weekly_power_impact = power_status_changed.groupby("week_start").size()

weekly_power_impact

In [ ]:
#Use deduplicated generation counts for the reporting calculation.
quill_user_week_final = quill_user_week_dedup.copy()

# Read the Quill active-user threshold from the configuration.
for tool in config["tools"]:
    if tool["tool_id"] == "quill":
        quill_active_threshold = tool["wau_threshold"]

# Flag employee-weeks that meet each usage threshold.
quill_user_week_final["is_active"] = (quill_user_week_final["generation_count_dedup"] >= quill_active_threshold)

quill_user_week_final["is_power"] = (quill_user_week_final["generation_count_dedup"] >= quill_power_threshold)

quill_user_week_final.head(10)

In [ ]:
# Count active and power users in each week by summing Boolean flags.
quill_weekly = quill_user_week_final.groupby("week_start")[["is_active", "is_power"]].sum()

# Rename the totals to the required reporting metric names.
quill_weekly = quill_weekly.rename(columns={"is_active": "wau", "is_power": "wpu"}).reset_index()

quill_weekly

In [ ]:
# Calculate the proportion of active users who are power users.
# Leave the rate empty when WAU is zero to avoid division by zero.
quill_weekly["wpu_rate"] = (quill_weekly["wpu"]/ quill_weekly["wau"].where(quill_weekly["wau"] > 0)).round(4)

quill_weekly

In [ ]:
# Identify the tool represented by these metrics.
quill_weekly["tool_id"] = "quill"

# Mark weeks that have source events before business filtering.
quill_weekly["measured"] = quill_weekly["week_start"].isin(quill_weekly_coverage.index)

# Arrange columns in the order required by the brief.
quill_weekly = quill_weekly[["week_start", "tool_id", "wau", "wpu", "wpu_rate", "measured"]]

quill_weekly

# ATLAS EDA

In [ ]:
# Inspect the first Atlas delivery without assuming a header row.
atlas_preview = pd.read_excel(data_dir / "atlas" / "atlas_usage_20260817.xlsx",sheet_name="Usage",header=None)

# Display the first eight rows to inspect the report layout.
atlas_preview.head(8)

In [ ]:
# Load the first Atlas delivery using the third row as column names.

atlas_first = pd.read_excel(data_dir / "atlas" / "atlas_usage_20260817.xlsx",sheet_name="Usage",header=2)

# Display the first five data rows.

atlas_first.head()

In [ ]:
# Inspect column names, non-null counts, and data types.

atlas_first.info()

In [ ]:
# Find and sort all Atlas usage Excel files.
atlas_files = sorted((data_dir / "atlas").glob("atlas_usage_*.xlsx"))

# Display the number of files and their names.
print("Atlas files found:", len(atlas_files))

for file in atlas_files:
    print(file.name)

In [ ]:
# Store each Atlas delivery as a separate DataFrame.
atlas_tables = []

for file in atlas_files:
    # Read the usage data using the third row as column names.
    atlas_table = pd.read_excel(file, sheet_name="Usage", header=2)

    # Preserve the source filename for tracing each record.
    atlas_table["source_file"] = file.name

    # Add this delivery to the list.
    atlas_tables.append(atlas_table)

# Display a sample from the first delivery.
atlas_tables[0].head()

In [ ]:
# Combine all Atlas deliveries into one DataFrame.
atlas_raw = pd.concat(atlas_tables, ignore_index=True)

# Display the total number of rows and a sample.
print("Total Atlas rows:", len(atlas_raw))
atlas_raw.head()

In [ ]:
# Count rows by source delivery and reporting week.
atlas_delivery_summary = (atlas_raw.groupby(["source_file", "Week Ending"]).size().reset_index(name="row_count"))

atlas_delivery_summary

In [ ]:
# Preserve the combined source data before cleaning.
atlas_clean = atlas_raw.copy()

# Extract the delivery date from each source filename.
atlas_clean["delivery_date"] = pd.to_datetime(atlas_clean["source_file"].str.replace("atlas_usage_", "", regex=False).str.replace(".xlsx", "", regex=False),format="%Y%m%d")

# Display each source file and its extracted delivery date once.
atlas_clean[["source_file", "delivery_date"]].drop_duplicates()

In [ ]:
# Find the latest delivery date for each reporting week.

atlas_clean["latest_delivery_date"] = (atlas_clean.groupby("Week Ending")["delivery_date"].transform("max"))

# Inspect the delivery dates against the latest date for each week.

atlas_clean[["Week Ending", "source_file", "delivery_date", "latest_delivery_date"]].drop_duplicates()

In [ ]:
# Retain only the latest delivery for each reporting week.

atlas_latest = atlas_clean[atlas_clean["delivery_date"] == atlas_clean["latest_delivery_date"]].copy()

# Check how many rows were retained and excluded.
print("Rows retained:", len(atlas_latest))
print("Superseded rows excluded:", len(atlas_clean) - len(atlas_latest))


In [ ]:
# Standardise employee IDs for consistent matching with the HR roster.

atlas_latest["User ID"] = (atlas_latest["User ID"].str.strip().str.upper())

# Display the first five standardised IDs.
atlas_latest["User ID"].head()

In [ ]:
# Inspect all repeated employee-week records in the latest deliveries.

atlas_repeated_users = atlas_latest[atlas_latest.duplicated(subset=["User ID", "Week Ending"], keep=False)]

print("Rows with repeated employee-week combinations:", len(atlas_repeated_users))
atlas_repeated_users.head(10)

In [ ]:
# Count missing values in the fields required for Atlas metrics.

atlas_latest[["User ID", "Week Ending", "Queries Run", "Active Days"]].isna().sum()

In [ ]:
# Inspect the distribution and range of the usage measures.
atlas_latest[["Queries Run", "Active Days"]].describe()

In [ ]:
# Inspect records with queries recorded but no active days.

atlas_usage_conflicts = atlas_latest[(atlas_latest["Queries Run"] > 0)& (atlas_latest["Active Days"] == 0)]

print("Rows with queries but no active days:", len(atlas_usage_conflicts))
atlas_usage_conflicts.head(10)

In [ ]:
# Check which weekdays appear in the vendor's week-ending dates.
atlas_latest["Week Ending"].dt.day_name().value_counts()

In [ ]:
# Convert Saturday week-ending dates to Sunday week starts in UTC.

atlas_latest["week_start"] = (pd.to_datetime(atlas_latest["Week Ending"], utc=True).dt.normalize()- pd.Timedelta(days=6))

# Display the distinct week-ending and week-start pairs.
atlas_latest[["Week Ending", "week_start"]].drop_duplicates()

In [ ]:
# Retain Atlas records within the configured reporting window.
atlas_window = atlas_latest[(atlas_latest["week_start"] >= first_week)& (atlas_latest["week_start"] <= last_week)].copy()

# Record source coverage before filtering to Advisory employees.
atlas_weekly_coverage = atlas_window.groupby("week_start").size()

atlas_weekly_coverage

In [ ]:
# Retain employees in the latest eligible Advisory roster.

atlas_advisory = atlas_window[atlas_window["User ID"].isin(hr_advisory["Employee_ID"])].copy()

# Display the retained row count and a sample.
print("Atlas Advisory employee-week rows:", len(atlas_advisory))
atlas_advisory.head()

In [ ]:
# Read the Atlas user thresholds from the configuration.
for tool in config["tools"]:
    if tool["tool_id"] == "atlas":
        atlas_active_threshold = tool["wau_threshold"]
        atlas_power_threshold = tool["wpu_threshold"]

# Display the thresholds used to classify Atlas users.
print("Active-user threshold (active days):", atlas_active_threshold)
print("Power-user threshold (queries run):", atlas_power_threshold)

In [ ]:
# Flag employee-weeks that meet the Atlas activity threshold.
atlas_advisory["is_active"] = (atlas_advisory["Active Days"] >= atlas_active_threshold)

# Flag employee-weeks that meet the Atlas power-user threshold.
atlas_advisory["is_power"] = (atlas_advisory["Queries Run"] >= atlas_power_threshold)

# Display the usage measures and classification flags.
atlas_advisory[
    ["week_start", "User ID", "Queries Run", "Active Days", "is_active", "is_power"]
].head(10)

In [ ]:
# Count active and power users for each reporting week.

atlas_weekly = (atlas_advisory.groupby("week_start")[["is_active", "is_power"]].sum().rename(columns={"is_active": "wau", "is_power": "wpu"}).reset_index())

atlas_weekly

In [ ]:
# Calculate the weekly share of active users who are power users.

atlas_weekly["wpu_rate"] = (atlas_weekly["wpu"] / atlas_weekly["wau"].where(atlas_weekly["wau"] > 0)).round(4)     # Rounding till 4
atlas_weekly

In [ ]:
# Identify the tool and mark weeks with Atlas source records.
atlas_weekly["tool_id"] = "atlas"

atlas_weekly["measured"] = atlas_weekly["week_start"].isin(atlas_weekly_coverage.index)

# Arrange the columns in the required output order.
atlas_weekly = atlas_weekly[["week_start", "tool_id", "wau", "wpu", "wpu_rate", "measured"]]

atlas_weekly

In [ ]:
# Build the full reporting calendar from the configured date range.

reporting_weeks = pd.date_range(start=first_week,end=last_week,freq="W-SUN")

reporting_weeks

In [ ]:
# Keep every reporting week and attach the available Atlas metrics.

atlas_full = pd.DataFrame({"week_start": reporting_weeks}).merge(atlas_weekly[["week_start", "wau", "wpu", "wpu_rate"]],on="week_start",how="left")


# Mark source coverage for every reporting week.
atlas_full["tool_id"] = "atlas"
atlas_full["measured"] = atlas_full["week_start"].isin(atlas_weekly_coverage.index)

# Fill missing counts with zero only for weeks with source coverage.
for column in ["wau", "wpu"]:
    atlas_full.loc[atlas_full["measured"], column] = (atlas_full.loc[atlas_full["measured"], column].fillna(0))
    atlas_full[column] = atlas_full[column].astype("Int64")

atlas_full[["week_start", "tool_id", "wau", "wpu", "wpu_rate", "measured"]]

In [ ]:
# Keep every reporting week and attach the available Quill metrics.
quill_full = pd.DataFrame({"week_start": reporting_weeks}).merge(quill_weekly[["week_start", "wau", "wpu", "wpu_rate"]],on="week_start",how="left")

# Mark source coverage for every reporting week.
quill_full["tool_id"] = "quill"
quill_full["measured"] = quill_full["week_start"].isin(quill_weekly_coverage.index)

# Fill missing counts with zero only for weeks with source coverage.
for column in ["wau", "wpu"]:
    quill_full.loc[quill_full["measured"], column] = (quill_full.loc[quill_full["measured"], column].fillna(0))
    quill_full[column] = quill_full[column].astype("Int64")

quill_full[["week_start", "tool_id", "wau", "wpu", "wpu_rate", "measured"]]

In [ ]:
# Select the same employee-week fields from both tools.
user_week_columns = ["week_start", "user_id", "is_active", "is_power"]

quill_flags = quill_user_week_final[user_week_columns]
atlas_flags = atlas_advisory.rename(columns={"User ID": "user_id"})[user_week_columns]

# Stack the employee-week flags from both tools.
any_user_rows = pd.concat([quill_flags, atlas_flags], ignore_index=True)

any_user_rows.head()

In [ ]:
# Combine tool-level flags into one row per employee per week.
any_user_week = (any_user_rows.groupby(["week_start", "user_id"])[["is_active", "is_power"]].any().reset_index())

any_user_week.head()

In [ ]:
# Count distinct active and power users across available tools by week.
any_weekly = (any_user_week.groupby("week_start")[["is_active", "is_power"]].sum().rename(columns={"is_active": "wau", "is_power": "wpu"})
    .reset_index())

any_weekly

In [ ]:
# Align the combined counts to the full reporting calendar.
any_full = pd.DataFrame({"week_start": reporting_weeks}).merge(any_weekly,on="week_start",how="left")

# Track source coverage separately for each tool.
quill_available = any_full["week_start"].isin(quill_weekly_coverage.index)
atlas_available = any_full["week_start"].isin(atlas_weekly_coverage.index)

# Mark full coverage and retain counts from available tools.
any_full["tool_id"] = "ANY"
any_full["measured"] = quill_available & atlas_available
any_source_available = quill_available | atlas_available

for column in ["wau", "wpu"]:
    any_full[column] = (any_full[column].fillna(0).where(any_source_available).astype("Int64"))

any_full

In [ ]:
# Calculate the power-user rate from the observed combined counts.
any_full["wpu_rate"] = (any_full["wpu"] / any_full["wau"].where(any_full["wau"] > 0)).round(4)

# Arrange the columns in the required output order.
any_full = any_full[["week_start", "tool_id", "wau", "wpu", "wpu_rate", "measured"]]

any_full

In [ ]:
# Combine the three summaries and select the required report columns.
report_columns = ["week_start", "tool_id", "wau", "wpu", "wpu_rate", "measured"]

weekly_tool_metrics = pd.concat([quill_full, atlas_full, any_full],ignore_index=True)[report_columns]

# Display the row count and the combined report.
print("Total report rows:", len(weekly_tool_metrics))
weekly_tool_metrics

In [ ]:
# Build the expected combinations of reporting weeks and tool IDs.
expected_rows = pd.MultiIndex.from_product([reporting_weeks, ["quill", "atlas", "ANY"]],names=["week_start", "tool_id"])

# Check that every expected combination appears exactly once.
actual_rows = weekly_tool_metrics.set_index(["week_start", "tool_id"]).index      # used assert for showing error message
assert actual_rows.sort_values().equals(expected_rows.sort_values()), (
    "The report has missing, duplicate, or unexpected week-tool rows."
)

# Check that power-user counts do not exceed active-user counts.
reported_counts = weekly_tool_metrics.dropna(subset=["wau", "wpu"])
assert (reported_counts["wpu"] <= reported_counts["wau"]).all(), (
    "A power-user count exceeds its active-user count."
)

print("Validation passed.")

In [ ]:
# Create the output folder if needed.
output_dir = project_dir / "out"
output_dir.mkdir(exist_ok=True)
output_file = output_dir / "weekly_tool_metrics.csv"

# Export the report with ISO dates, four decimal places, and blank missing values.
report_export = weekly_tool_metrics.copy()
report_export["wpu_rate"] = report_export["wpu_rate"].astype(float)

# Use lowercase Boolean values in the CSV.
report_export["measured"] = report_export["measured"].map({True: "true", False: "false"})

report_export.to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d",
    float_format="%.4f",
    na_rep="")

print("CSV saved to:", output_file)

In [ ]:
# Verify the exported CSV using its original text values.
saved_report = pd.read_csv(output_file, dtype=str, keep_default_na=False)

assert saved_report.columns.tolist() == report_columns, "Unexpected CSV headers."
assert len(saved_report) == len(weekly_tool_metrics), "Unexpected CSV row count."

# Display the week with partial source coverage.
saved_report[saved_report["week_start"] == "2026-09-20"]